# Cross-Sectional Factor Neutral with Funding Squeeze

## Thesis

This model exploits **liquidation cascade mechanics** in crypto futures. When leveraged positions crowd one side of a market (visible via extreme funding rates), the eventual squeeze/cascade creates predictable cross-sectional return dispersion.

### Structural Edge
- Funding rates are publicly observable → the crowd's positioning is visible
- Extreme funding = overleveraged crowd → liquidation cascade WILL happen
- By going contrarian on funding AND ranking by momentum/TFI factors, we select which assets will benefit most from the cascade
- TOTAL3/TOTAL2 ratio gates alt-season vs alt-winter → only trade when dispersion regime is favorable

### Hypotheses

| H | Hypothesis | GO | NO-GO |
|---|-----------|-----|-------|
| H1 | Momentum factor IC > 0 (cross-sectional momentum works in crypto) | mean IC > 0.03, t-stat > 2 | mean IC < 0.01 |
| H2 | Extreme funding predicts reversals across universe | Mean fwd return at \|funding_z\| > 1.5 is contrarian with WR > 55% | WR < 52% |
| H3 | Funding filter improves L/S portfolio vs unfiltered | Sharpe improvement > 20% | < 5% |
| H4 | Net portfolio Sharpe (after 4bps costs) > 0.5 annualized | Net Sharpe > 0.5 | < 0.3 |
| H5 | Alt-season regime gate improves timing | Portfolio in alt-season beats alt-winter by > 50% Sharpe | No difference |

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import time
from datetime import datetime, timezone
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from binance.um_futures import UMFutures

plt.style.use('dark_background')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)

BASE_URL = 'https://fapi.binance.com'
client = UMFutures()

UNIVERSE = [
    'BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'SOLUSDT', 'XRPUSDT',
    'DOGEUSDT', 'ADAUSDT', 'AVAXUSDT', 'DOTUSDT', 'LINKUSDT',
    'MATICUSDT', 'UNIUSDT', 'ATOMUSDT', 'LTCUSDT', 'NEARUSDT',
    'APTUSDT', 'ARBUSDT', 'OPUSDT', 'SUIUSDT', 'INJUSDT',
]

# ── OHLCV fetch (reused from hypothesis_liquidity_models.ipynb) ──
_ALL_COLS = [
    'timestamp', 'open', 'high', 'low', 'close', 'volume', 'close_time',
    'quote_vol', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore',
]
_KEEP_COLS = ['timestamp', 'open', 'high', 'low', 'close', 'volume',
              'quote_vol', 'trades', 'taker_buy_base', 'taker_buy_quote']
_MAX_LIMIT = 1500

def fetch_ohlcv_full(
    symbol: str,
    timeframe: str,
    start_date: str = '2025-06-01',
    end_date: str | None = None,
) -> pd.DataFrame:
    """Fetch OHLCV + taker fields from Binance Futures."""
    since = int(datetime.strptime(start_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000)
    if end_date:
        end = int(datetime.strptime(end_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000)
    else:
        end = int(datetime.now(timezone.utc).timestamp() * 1000)
    frames = []
    cursor = since
    while cursor < end:
        lines = client.klines(symbol, timeframe, startTime=cursor, endTime=end, limit=_MAX_LIMIT)
        if not lines:
            break
        df = pd.DataFrame(lines, columns=_ALL_COLS)[_KEEP_COLS]
        for c in _KEEP_COLS:
            df[c] = pd.to_numeric(df[c], errors='coerce')
        frames.append(df)
        last_ts = int(df['timestamp'].iloc[-1])
        if last_ts <= cursor:
            break
        cursor = last_ts + 1
        if len(lines) < _MAX_LIMIT:
            break
    if not frames:
        return pd.DataFrame(columns=_KEEP_COLS)
    result = pd.concat(frames, ignore_index=True)
    result = result.drop_duplicates('timestamp').sort_values('timestamp').reset_index(drop=True)
    result['dt'] = pd.to_datetime(result['timestamp'], unit='ms', utc=True)
    result = result.set_index('dt')
    return result

# ── Funding rate fetch (reused from hypothesis_liquidity_models.ipynb) ──
def fetch_funding_rate(
    symbol: str,
    start_date: str = '2025-06-01',
    end_date: str | None = None,
) -> pd.DataFrame:
    """Fetch funding rate history (every 8h) with pagination."""
    since = int(datetime.strptime(start_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000)
    end = int(datetime.strptime(end_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000) if end_date else int(datetime.now(timezone.utc).timestamp() * 1000)
    frames = []
    cursor = since
    while cursor < end:
        resp = requests.get(f'{BASE_URL}/fapi/v1/fundingRate', params={
            'symbol': symbol, 'startTime': cursor, 'endTime': end, 'limit': 1000
        })
        data = resp.json()
        if not data:
            break
        df = pd.DataFrame(data)
        frames.append(df)
        cursor = int(df['fundingTime'].iloc[-1]) + 1
        if len(data) < 1000:
            break
        time.sleep(0.2)
    if not frames:
        return pd.DataFrame()
    result = pd.concat(frames, ignore_index=True)
    result['fundingRate'] = result['fundingRate'].astype(float)
    result['fundingTime'] = pd.to_datetime(result['fundingTime'].astype(int), unit='ms', utc=True)
    if 'markPrice' in result.columns:
        result['markPrice'] = pd.to_numeric(result['markPrice'], errors='coerce')
    result = result.set_index('fundingTime').sort_index()
    return result

# ── TV CSV loader ──
def load_tv_csv(name: str) -> pd.DataFrame:
    """Load TradingView index CSV from data/tv_index/."""
    path = f'../data/tv_index/{name}_1h.csv'
    df = pd.read_csv(path)
    df['dt'] = pd.to_datetime(df['datetime'], utc=True)
    df = df.set_index('dt').sort_index()
    return df

print('Setup complete')
print(f'Universe: {len(UNIVERSE)} symbols')

In [ ]:
# ── Cell 3: Fetch universe OHLCV (4h) ──────────────────────────────
def fetch_universe_data(symbols, timeframe='4h', start_date='2025-06-01'):
    """Fetch OHLCV + taker for entire universe. Returns dict of DataFrames."""
    universe = {}
    skipped = []
    for i, sym in enumerate(symbols):
        print(f'  [{i+1}/{len(symbols)}] Fetching {sym}...', end=' ')
        try:
            df = fetch_ohlcv_full(sym, timeframe, start_date)
            if len(df) > 200:
                universe[sym] = df
                print(f'{len(df)} bars')
            else:
                skipped.append(sym)
                print(f'SKIP (only {len(df)} bars)')
        except Exception as e:
            skipped.append(sym)
            print(f'SKIP ({e})')
        time.sleep(0.2)
    print(f'\nUniverse loaded: {len(universe)}/{len(symbols)} symbols')
    if skipped:
        print(f'Skipped: {skipped}')
    return universe

print('Fetching universe OHLCV data (4h bars)...')
universe_data = fetch_universe_data(UNIVERSE, timeframe='4h', start_date='2025-06-01')

# Show sample
sample_sym = list(universe_data.keys())[0]
print(f'\nSample ({sample_sym}): {len(universe_data[sample_sym])} bars')
print(universe_data[sample_sym].tail(3))

In [ ]:
# ── Cell 4: Fetch universe funding rates ──────────────────────────
def fetch_universe_funding(symbols, start_date='2025-06-01'):
    """Fetch funding rates for entire universe."""
    funding = {}
    skipped = []
    for i, sym in enumerate(symbols):
        print(f'  [{i+1}/{len(symbols)}] Funding {sym}...', end=' ')
        try:
            fr = fetch_funding_rate(sym, start_date)
            if len(fr) > 50:
                funding[sym] = fr
                print(f'{len(fr)} records')
            else:
                skipped.append(sym)
                print(f'SKIP (only {len(fr)} records)')
        except Exception as e:
            skipped.append(sym)
            print(f'SKIP ({e})')
        time.sleep(0.2)
    print(f'\nFunding loaded: {len(funding)}/{len(symbols)} symbols')
    if skipped:
        print(f'Skipped: {skipped}')
    return funding

print('Fetching universe funding rates...')
universe_funding = fetch_universe_funding(list(universe_data.keys()), start_date='2025-06-01')

# Show sample
sample_sym = list(universe_funding.keys())[0]
print(f'\nSample ({sample_sym}):')
print(universe_funding[sample_sym].tail(3))

In [ ]:
# ── Cell 5: Load TV indices & compute alt-season regime ──────────
btc_d = load_tv_csv('BTC_D')
total2 = load_tv_csv('TOTAL2')
total3 = load_tv_csv('TOTAL3')
print(f'BTC.D:  {len(btc_d)} bars ({btc_d.index[0].date()} to {btc_d.index[-1].date()})')
print(f'TOTAL2: {len(total2)} bars ({total2.index[0].date()} to {total2.index[-1].date()})')
print(f'TOTAL3: {len(total3)} bars ({total3.index[0].date()} to {total3.index[-1].date()})')

def compute_alt_season_regime(total3_df, total2_df):
    """TOTAL3/TOTAL2 ratio as alt-season regime gate.
    
    Rising ratio (above 48h SMA) → alt-season → more cross-sectional dispersion
    Falling ratio (below 48h SMA) → alt-winter → less dispersion
    """
    t3 = total3_df[['close']].rename(columns={'close': 'total3'})
    t2 = total2_df[['close']].rename(columns={'close': 'total2'})
    merged = t3.join(t2, how='inner')
    merged['alt_ratio'] = merged['total3'] / merged['total2']
    merged['alt_ratio_sma'] = merged['alt_ratio'].rolling(48).mean()
    merged['alt_season'] = merged['alt_ratio'] > merged['alt_ratio_sma']
    merged['alt_ratio_roc'] = merged['alt_ratio'].pct_change(24)
    return merged

alt_regime = compute_alt_season_regime(total3, total2)
print(f'\nAlt-season regime: {len(alt_regime)} bars')
print(f'Alt-season fraction: {alt_regime["alt_season"].mean():.2%}')

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(alt_regime.index, alt_regime['alt_ratio'], label='TOTAL3/TOTAL2', alpha=0.8)
axes[0].plot(alt_regime.index, alt_regime['alt_ratio_sma'], label='48h SMA', alpha=0.6, linestyle='--')
axes[0].set_title('Alt-Season Ratio (TOTAL3/TOTAL2)')
axes[0].legend()
alt_s = alt_regime['alt_season'].astype(int)
axes[1].fill_between(alt_regime.index, 0, alt_s, alpha=0.3, color='lime', label='Alt-Season')
axes[1].fill_between(alt_regime.index, 0, 1 - alt_s, alpha=0.3, color='red', label='Alt-Winter')
axes[1].set_title('Regime')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 6: Compute cross-sectional factors ──────────────────────
def compute_cross_sectional_factors(universe, funding):
    """Compute factor scores for each symbol at each 4h timestamp.
    
    Factors:
      momentum: 24-bar (96h) return
      tfi: taker_buy_base / volume, EMA(5)-smoothed
      funding_z: (funding_rate - rolling_mean_90) / rolling_std_90
    
    Returns panel DataFrame indexed by (timestamp) with 'symbol' column.
    """
    panels = []
    for sym, df in universe.items():
        factors = pd.DataFrame(index=df.index)
        factors['symbol'] = sym
        factors['close'] = df['close']
        
        # Factor 1: Momentum (24 bars × 4h = 96h)
        factors['momentum'] = df['close'].pct_change(24)
        
        # Factor 2: TFI (taker flow imbalance)
        factors['tfi'] = (df['taker_buy_base'] / df['volume'].replace(0, np.nan)).ewm(span=5).mean()
        
        # Factor 3: Funding rate z-score
        if sym in funding:
            fr = funding[sym][['fundingRate']].copy()
            # Resample funding (8h) to 4h bars via forward-fill
            fr_resampled = fr.resample('4h').ffill()
            factors = factors.join(fr_resampled, how='left')
            factors['fundingRate'] = factors['fundingRate'].ffill()
            # 90 bars at 8h ≈ 30 days
            fr_mean = factors['fundingRate'].rolling(90, min_periods=30).mean()
            fr_std = factors['fundingRate'].rolling(90, min_periods=30).std()
            factors['funding_z'] = (factors['fundingRate'] - fr_mean) / fr_std.replace(0, np.nan)
        else:
            factors['fundingRate'] = np.nan
            factors['funding_z'] = np.nan
        
        # Forward 4h return (for evaluation)
        factors['fwd_ret_4h'] = df['close'].pct_change(1).shift(-1)
        
        panels.append(factors)
    
    panel = pd.concat(panels)
    panel.index.name = 'timestamp'
    print(f'Panel: {len(panel)} rows, {panel["symbol"].nunique()} symbols')
    print(f'Factor coverage:')
    for col in ['momentum', 'tfi', 'funding_z']:
        pct = panel[col].notna().mean()
        print(f'  {col}: {pct:.1%} non-null')
    return panel

panel = compute_cross_sectional_factors(universe_data, universe_funding)
panel.head()

In [ ]:
# ── Cell 7: Cross-sectional diagnostics ──────────────────────────
# Factor distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ['momentum', 'tfi', 'funding_z']):
    vals = panel[col].dropna()
    ax.hist(vals.clip(vals.quantile(0.01), vals.quantile(0.99)), bins=60, alpha=0.7)
    ax.set_title(f'{col} distribution')
    ax.axvline(vals.median(), color='yellow', linestyle='--', label=f'median={vals.median():.4f}')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Factor correlation matrix
factor_cols = ['momentum', 'tfi', 'funding_z']
corr = panel[factor_cols].corr()
print('Factor correlation matrix:')
print(corr.round(3))

# Factor IC: rank correlation of each factor vs forward 4h return
print('\n--- Factor Information Coefficient (IC) ---')
print('Rank correlation of factor vs fwd 4h return, computed per timestamp then averaged')
timestamps = panel.index.unique()
ic_records = {col: [] for col in factor_cols}

for ts in timestamps:
    group = panel.loc[ts]
    if isinstance(group, pd.Series):  # single row
        continue
    if len(group) < 8:
        continue
    for col in factor_cols:
        valid = group.dropna(subset=[col, 'fwd_ret_4h'])
        if len(valid) < 8:
            continue
        rho, _ = stats.spearmanr(valid[col], valid['fwd_ret_4h'])
        ic_records[col].append(rho)

print(f'\n{"Factor":<12} {"Mean IC":>10} {"Std IC":>10} {"t-stat":>10} {"IC > 0 %":>10}')
print('-' * 55)
for col in factor_cols:
    ics = np.array(ic_records[col])
    if len(ics) > 0:
        mean_ic = ics.mean()
        std_ic = ics.std()
        t = mean_ic / (std_ic / np.sqrt(len(ics))) if std_ic > 0 else 0
        pct_pos = (ics > 0).mean()
        print(f'{col:<12} {mean_ic:>10.4f} {std_ic:>10.4f} {t:>10.2f} {pct_pos:>10.1%}')

In [ ]:
# ── Cell 8: H1 — Cross-sectional momentum factor IC ─────────────
print('='*60)
print('H1: Cross-sectional momentum factor IC > 0')
print('='*60)
print('Test: rank correlation of momentum rank vs fwd 4h return')
print()

mom_ics = np.array(ic_records['momentum'])
mean_ic = mom_ics.mean()
std_ic = mom_ics.std()
t_stat = mean_ic / (std_ic / np.sqrt(len(mom_ics))) if std_ic > 0 else 0

print(f'Mean momentum IC:  {mean_ic:.4f}')
print(f'Std IC:            {std_ic:.4f}')
print(f't-stat:            {t_stat:.2f}')
print(f'N periods:         {len(mom_ics)}')
print(f'IC > 0 fraction:   {(mom_ics > 0).mean():.1%}')
print()

# IC time series plot
fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(range(len(mom_ics)), mom_ics, alpha=0.3, width=1)
ax.axhline(0, color='white', linewidth=0.5)
ax.axhline(mean_ic, color='lime', linewidth=1.5, label=f'Mean IC = {mean_ic:.4f}')
ax.set_title('Momentum Factor IC Time Series')
ax.set_xlabel('Rebalance period')
ax.set_ylabel('Rank IC')
ax.legend()
plt.tight_layout()
plt.show()

h1_go = mean_ic > 0.03 and t_stat > 2.0
h1_nogo = mean_ic < 0.01
h1_verdict = 'GO' if h1_go else ('NO-GO' if h1_nogo else 'BORDERLINE')
print(f'\nH1 VERDICT: {h1_verdict}')
print(f'  GO criteria: mean IC > 0.03 AND t-stat > 2  →  {mean_ic:.4f} > 0.03 = {mean_ic > 0.03}, t={t_stat:.2f} > 2 = {t_stat > 2.0}')
print(f'  NO-GO criteria: mean IC < 0.01  →  {mean_ic:.4f} < 0.01 = {mean_ic < 0.01}')

In [ ]:
# ── Cell 9: H2 — Funding extremes predict reversals ──────────────
print('='*60)
print('H2: Extreme funding predicts reversals across universe')
print('='*60)
print('Test: conditional returns at |funding_z| > 1.5')
print()

valid = panel.dropna(subset=['funding_z', 'fwd_ret_4h'])

# High funding (longs crowded) → expect negative fwd return (reversal DOWN)
high_funding = valid[valid['funding_z'] > 1.5]
high_fwd = high_funding['fwd_ret_4h']
high_wr_reversal = (high_fwd < 0).mean()  # Win rate for contrarian short

# Low funding (shorts crowded) → expect positive fwd return (reversal UP)
low_funding = valid[valid['funding_z'] < -1.5]
low_fwd = low_funding['fwd_ret_4h']
low_wr_reversal = (low_fwd > 0).mean()  # Win rate for contrarian long

# Combined
combined_wr = np.nan
if len(high_fwd) + len(low_fwd) > 0:
    correct_high = (high_fwd < 0).sum()
    correct_low = (low_fwd > 0).sum()
    combined_wr = (correct_high + correct_low) / (len(high_fwd) + len(low_fwd))

print(f'HIGH funding (z > 1.5): {len(high_funding)} events')
print(f'  Mean fwd 4h return:   {high_fwd.mean():.4%}')
print(f'  Contrarian short WR:  {high_wr_reversal:.1%}')
print()
print(f'LOW funding (z < -1.5): {len(low_funding)} events')
print(f'  Mean fwd 4h return:   {low_fwd.mean():.4%}')
print(f'  Contrarian long WR:   {low_wr_reversal:.1%}')
print()
print(f'COMBINED contrarian WR: {combined_wr:.1%}')

# Unconditional for comparison
uncond = valid['fwd_ret_4h']
print(f'Unconditional mean fwd: {uncond.mean():.4%}')

# Funding z-score quintile analysis
valid_q = valid.copy()
valid_q['fz_q'] = pd.qcut(valid_q['funding_z'], 5, labels=['Q1(low)', 'Q2', 'Q3', 'Q4', 'Q5(high)'], duplicates='drop')
q_stats = valid_q.groupby('fz_q')['fwd_ret_4h'].agg(['mean', 'std', 'count'])
print('\nFunding Z-Score Quintile Analysis:')
print(q_stats.round(5))

fig, ax = plt.subplots(figsize=(8, 4))
q_stats['mean'].plot.bar(ax=ax, color=['lime', 'green', 'gray', 'red', 'darkred'], alpha=0.7)
ax.set_title('Mean Fwd 4h Return by Funding Z-Score Quintile')
ax.set_ylabel('Mean fwd return')
ax.axhline(0, color='white', linewidth=0.5)
plt.tight_layout()
plt.show()

h2_go = combined_wr > 0.55 if not np.isnan(combined_wr) else False
h2_nogo = combined_wr < 0.52 if not np.isnan(combined_wr) else True
h2_verdict = 'GO' if h2_go else ('NO-GO' if h2_nogo else 'BORDERLINE')
print(f'\nH2 VERDICT: {h2_verdict}')
print(f'  GO criteria: combined contrarian WR > 55%  →  {combined_wr:.1%} > 55% = {h2_go}')
print(f'  NO-GO criteria: combined contrarian WR < 52%  →  {combined_wr:.1%} < 52% = {h2_nogo}')

In [ ]:
# ── Cell 10: H3 — Contrarian funding filter adds value ──────────
print('='*60)
print('H3: Contrarian funding filter adds value vs unfiltered')
print('='*60)
print()

def construct_ls_portfolio(panel_df, n_long=5, n_short=5, use_funding_filter=True):
    """Rank assets cross-sectionally and construct long/short portfolio.
    
    Composite score = mean of momentum_rank + tfi_rank.
    Top N = long candidates, Bottom N = short candidates.
    
    If use_funding_filter:
      - Only long where funding_z < -0.5 (shorts crowded → squeeze UP)
      - Only short where funding_z > 0.5 (longs crowded → squeeze DOWN)
    """
    portfolios = []
    timestamps = panel_df.index.unique()
    
    for ts in timestamps:
        group = panel_df.loc[ts]
        if isinstance(group, pd.Series):
            continue
        g = group.dropna(subset=['momentum', 'tfi'])
        if len(g) < 10:
            continue
        
        # Rank each factor within cross-section
        g = g.copy()
        g['mom_rank'] = g['momentum'].rank(pct=True)
        g['tfi_rank'] = g['tfi'].rank(pct=True)
        g['composite'] = g[['mom_rank', 'tfi_rank']].mean(axis=1)
        g = g.sort_values('composite', ascending=False)
        
        if use_funding_filter:
            # Top composite + negative funding → long (shorts crowded)
            long_cands = g.head(n_long * 2)
            longs = long_cands[long_cands['funding_z'] < -0.5].head(n_long)
            # Bottom composite + positive funding → short (longs crowded)
            short_cands = g.tail(n_short * 2)
            shorts = short_cands[short_cands['funding_z'] > 0.5].head(n_short)
        else:
            longs = g.head(n_long)
            shorts = g.tail(n_short)
        
        for _, row in longs.iterrows():
            portfolios.append({'ts': ts, 'symbol': row['symbol'], 'side': 'long',
                             'composite': row['composite'], 'funding_z': row.get('funding_z', np.nan)})
        for _, row in shorts.iterrows():
            portfolios.append({'ts': ts, 'symbol': row['symbol'], 'side': 'short',
                             'composite': row['composite'], 'funding_z': row.get('funding_z', np.nan)})
    
    return pd.DataFrame(portfolios)

# Build both filtered and unfiltered portfolios
pf_filtered = construct_ls_portfolio(panel, use_funding_filter=True)
pf_unfiltered = construct_ls_portfolio(panel, use_funding_filter=False)

print(f'Filtered portfolio:   {len(pf_filtered)} position-entries across {pf_filtered["ts"].nunique()} rebalances')
print(f'Unfiltered portfolio: {len(pf_unfiltered)} position-entries across {pf_unfiltered["ts"].nunique()} rebalances')

# -- Backtest helper --
def backtest_ls(portfolios_df, universe, fee_bps=4):
    """Backtest L/S portfolio with transaction costs."""
    results = []
    prev_positions = set()
    
    for ts in sorted(portfolios_df['ts'].unique()):
        pf = portfolios_df[portfolios_df['ts'] == ts]
        longs = pf[pf['side'] == 'long']['symbol'].tolist()
        shorts = pf[pf['side'] == 'short']['symbol'].tolist()
        
        long_ret = []
        for sym in longs:
            if sym in universe:
                idx = universe[sym].index.get_indexer([ts], method='nearest')[0]
                if idx + 1 < len(universe[sym]):
                    ret = (universe[sym]['close'].iloc[idx+1] / universe[sym]['close'].iloc[idx]) - 1
                    long_ret.append(ret)
        
        short_ret = []
        for sym in shorts:
            if sym in universe:
                idx = universe[sym].index.get_indexer([ts], method='nearest')[0]
                if idx + 1 < len(universe[sym]):
                    ret = (universe[sym]['close'].iloc[idx+1] / universe[sym]['close'].iloc[idx]) - 1
                    short_ret.append(-ret)
        
        gross = 0.0
        n_pos = 0
        if long_ret:
            gross += np.mean(long_ret)
            n_pos += len(long_ret)
        if short_ret:
            gross += np.mean(short_ret)
            n_pos += len(short_ret)
        
        # Transaction costs: count changed positions, fee_bps each way (2×fee_bps round-trip)
        current_positions = set(f'{s}_{side}' for s, side in zip(
            pf['symbol'].tolist(), pf['side'].tolist()))
        changed = len(current_positions - prev_positions) + len(prev_positions - current_positions)
        total_slots = max(len(current_positions) + len(prev_positions), 1)
        turnover_frac = changed / total_slots
        cost = turnover_frac * (fee_bps * 2) / 10000  # round-trip cost on turnover fraction
        prev_positions = current_positions
        
        net = gross - cost
        results.append({'ts': ts, 'gross': gross, 'net': net,
                       'n_long': len(longs), 'n_short': len(shorts),
                       'turnover': turnover_frac, 'n_changed': changed})
    
    return pd.DataFrame(results).set_index('ts')

bt_filtered = backtest_ls(pf_filtered, universe_data)
bt_unfiltered = backtest_ls(pf_unfiltered, universe_data)

# Annualize: 4h bars → 6 per day → ~2190 per year
bars_per_year = 6 * 365

def sharpe(series, bpy=bars_per_year):
    if series.std() == 0:
        return 0.0
    return series.mean() / series.std() * np.sqrt(bpy)

sr_filt_gross = sharpe(bt_filtered['gross'])
sr_filt_net = sharpe(bt_filtered['net'])
sr_unfilt_gross = sharpe(bt_unfiltered['gross'])
sr_unfilt_net = sharpe(bt_unfiltered['net'])

improvement = (sr_filt_net / sr_unfilt_net - 1) * 100 if sr_unfilt_net != 0 else np.nan

print(f'\n{"Metric":<25} {"Filtered":>12} {"Unfiltered":>12}')
print('-' * 52)
print(f'{"Gross Sharpe":<25} {sr_filt_gross:>12.3f} {sr_unfilt_gross:>12.3f}')
print(f'{"Net Sharpe":<25} {sr_filt_net:>12.3f} {sr_unfilt_net:>12.3f}')
print(f'{"Mean gross ret/bar":<25} {bt_filtered["gross"].mean():>12.5%} {bt_unfiltered["gross"].mean():>12.5%}')
print(f'{"Mean turnover":<25} {bt_filtered["turnover"].mean():>12.1%} {bt_unfiltered["turnover"].mean():>12.1%}')
print(f'{"N rebalances":<25} {len(bt_filtered):>12d} {len(bt_unfiltered):>12d}')
print(f'\nSharpe improvement (filtered vs unfiltered): {improvement:+.1f}%')

h3_go = improvement > 20 if not np.isnan(improvement) else False
h3_nogo = improvement < 5 if not np.isnan(improvement) else True
h3_verdict = 'GO' if h3_go else ('NO-GO' if h3_nogo else 'BORDERLINE')
print(f'\nH3 VERDICT: {h3_verdict}')
print(f'  GO criteria: Sharpe improvement > 20%  →  {improvement:+.1f}% > 20% = {h3_go}')
print(f'  NO-GO criteria: Sharpe improvement < 5%  →  {improvement:+.1f}% < 5% = {h3_nogo}')

In [ ]:
# ── Cell 11: construct_ls_portfolio (already defined above, show portfolio summary) ──
print('Portfolio composition summary (filtered):')
print(f'  Total rebalance periods: {pf_filtered["ts"].nunique()}')
long_counts = pf_filtered[pf_filtered['side'] == 'long'].groupby('ts').size()
short_counts = pf_filtered[pf_filtered['side'] == 'short'].groupby('ts').size()
print(f'  Avg longs per period:  {long_counts.mean():.1f} (min={long_counts.min()}, max={long_counts.max()})')
print(f'  Avg shorts per period: {short_counts.mean():.1f} (min={short_counts.min()}, max={short_counts.max()})')

# Most frequently held assets
freq_long = pf_filtered[pf_filtered['side']=='long']['symbol'].value_counts().head(10)
freq_short = pf_filtered[pf_filtered['side']=='short']['symbol'].value_counts().head(10)
print('\nMost frequent longs:')
for sym, cnt in freq_long.items():
    print(f'  {sym}: {cnt} periods')
print('\nMost frequent shorts:')
for sym, cnt in freq_short.items():
    print(f'  {sym}: {cnt} periods')

In [ ]:
# ── Cell 12: Portfolio backtest with transaction cost modeling ────
print('='*60)
print('Portfolio Backtest — Filtered L/S with 4bps Transaction Costs')
print('='*60)
print()

# Already computed bt_filtered in Cell 10, display detailed stats
bt = bt_filtered.copy()

cum_gross = (1 + bt['gross']).cumprod()
cum_net = (1 + bt['net']).cumprod()

# Max drawdown
def max_drawdown(equity):
    peak = equity.cummax()
    dd = (equity - peak) / peak
    return dd.min()

mdd_gross = max_drawdown(cum_gross)
mdd_net = max_drawdown(cum_net)

total_return_gross = cum_gross.iloc[-1] / cum_gross.iloc[0] - 1
total_return_net = cum_net.iloc[-1] / cum_net.iloc[0] - 1

months = (bt.index[-1] - bt.index[0]).total_seconds() / (30 * 86400)
ann_factor = 12 / months if months > 0 else 1

print(f'{"Metric":<30} {"Gross":>12} {"Net":>12}')
print('-' * 55)
print(f'{"Total return":<30} {total_return_gross:>12.2%} {total_return_net:>12.2%}')
print(f'{"Annualized Sharpe":<30} {sr_filt_gross:>12.3f} {sr_filt_net:>12.3f}')
print(f'{"Max drawdown":<30} {mdd_gross:>12.2%} {mdd_net:>12.2%}')
print(f'{"Mean return/bar":<30} {bt["gross"].mean():>12.5%} {bt["net"].mean():>12.5%}')
print(f'{"Std return/bar":<30} {bt["gross"].std():>12.5%} {bt["net"].std():>12.5%}')
print(f'{"Mean turnover":<30} {bt["turnover"].mean():>12.1%}')
print(f'{"Mean longs/bar":<30} {bt["n_long"].mean():>12.1f}')
print(f'{"Mean shorts/bar":<30} {bt["n_short"].mean():>12.1f}')
print(f'{"N rebalances":<30} {len(bt):>12d}')
print(f'{"Period":<30} {str(bt.index[0].date()):>12} → {str(bt.index[-1].date())}')

# Win rate
wr_gross = (bt['gross'] > 0).mean()
wr_net = (bt['net'] > 0).mean()
print(f'{"Win rate":<30} {wr_gross:>12.1%} {wr_net:>12.1%}')

# Profit factor
pf_g = bt['gross'][bt['gross'] > 0].sum() / abs(bt['gross'][bt['gross'] < 0].sum()) if (bt['gross'] < 0).any() else np.inf
pf_n = bt['net'][bt['net'] > 0].sum() / abs(bt['net'][bt['net'] < 0].sum()) if (bt['net'] < 0).any() else np.inf
print(f'{"Profit factor":<30} {pf_g:>12.2f} {pf_n:>12.2f}')

In [ ]:
# ── Cell 13: H4 — Net Sharpe after costs > 0.5 ──────────────────
print('='*60)
print('H4: Net Sharpe after costs > 0.5 annualized')
print('='*60)
print()

print(f'Net Sharpe (annualized): {sr_filt_net:.3f}')
print(f'Gross Sharpe:            {sr_filt_gross:.3f}')
print(f'Cost drag:               {(sr_filt_gross - sr_filt_net):.3f} Sharpe units')
print(f'Mean turnover per bar:   {bt_filtered["turnover"].mean():.1%}')
print()

# Sensitivity to fee level
print('Fee sensitivity:')
for fee in [0, 2, 4, 6, 8, 10]:
    bt_fee = backtest_ls(pf_filtered, universe_data, fee_bps=fee)
    sr_fee = sharpe(bt_fee['net'])
    print(f'  {fee} bps: Net Sharpe = {sr_fee:.3f}')

h4_go = sr_filt_net > 0.5
h4_nogo = sr_filt_net < 0.3
h4_verdict = 'GO' if h4_go else ('NO-GO' if h4_nogo else 'BORDERLINE')
print(f'\nH4 VERDICT: {h4_verdict}')
print(f'  GO criteria: Net Sharpe > 0.5  →  {sr_filt_net:.3f} > 0.5 = {h4_go}')
print(f'  NO-GO criteria: Net Sharpe < 0.3  →  {sr_filt_net:.3f} < 0.3 = {h4_nogo}')

In [ ]:
# ── Cell 14: H5 — Alt-season regime gate ─────────────────────────
print('='*60)
print('H5: Alt-season regime gate improves timing')
print('='*60)
print()

# Map alt-season regime to backtest timestamps
# Resample alt_regime from 1h to 4h
alt_regime_4h = alt_regime[['alt_season']].resample('4h').last().ffill()

bt_with_regime = bt_filtered.copy()
bt_with_regime = bt_with_regime.join(alt_regime_4h, how='left')
bt_with_regime['alt_season'] = bt_with_regime['alt_season'].ffill()

# Split performance by regime
bt_alt_season = bt_with_regime[bt_with_regime['alt_season'] == True]
bt_alt_winter = bt_with_regime[bt_with_regime['alt_season'] == False]

sr_season = sharpe(bt_alt_season['net']) if len(bt_alt_season) > 10 else np.nan
sr_winter = sharpe(bt_alt_winter['net']) if len(bt_alt_winter) > 10 else np.nan

print(f'{"Metric":<25} {"Alt-Season":>12} {"Alt-Winter":>12} {"Full":>12}')
print('-' * 65)
print(f'{"N bars":<25} {len(bt_alt_season):>12d} {len(bt_alt_winter):>12d} {len(bt_with_regime):>12d}')
print(f'{"Net Sharpe":<25} {sr_season:>12.3f} {sr_winter:>12.3f} {sr_filt_net:>12.3f}')
if len(bt_alt_season) > 0:
    print(f'{"Mean net ret/bar":<25} {bt_alt_season["net"].mean():>12.5%} {bt_alt_winter["net"].mean():>12.5%} {bt_filtered["net"].mean():>12.5%}')
    print(f'{"Win rate":<25} {(bt_alt_season["net"]>0).mean():>12.1%} {(bt_alt_winter["net"]>0).mean():>12.1%} {(bt_filtered["net"]>0).mean():>12.1%}')

# Sharpe improvement of alt-season over alt-winter
if sr_winter != 0 and not np.isnan(sr_winter) and not np.isnan(sr_season):
    regime_improvement = (sr_season / sr_winter - 1) * 100 if sr_winter > 0 else np.inf
    # Handle case where winter sharpe is negative
    if sr_winter < 0 and sr_season > 0:
        regime_improvement = np.inf  # clearly better
    elif sr_winter < 0 and sr_season < 0:
        regime_improvement = 0  # both bad
else:
    regime_improvement = np.nan

print(f'\nAlt-season Sharpe advantage: {regime_improvement:+.1f}%' if not np.isinf(regime_improvement) and not np.isnan(regime_improvement) else f'\nAlt-season Sharpe advantage: INF (winter is negative)')

h5_go = (not np.isnan(regime_improvement)) and (regime_improvement > 50 or np.isinf(regime_improvement))
h5_nogo = np.isnan(regime_improvement) or (regime_improvement < 10 and not np.isinf(regime_improvement))
h5_verdict = 'GO' if h5_go else ('NO-GO' if h5_nogo else 'BORDERLINE')
print(f'\nH5 VERDICT: {h5_verdict}')
print(f'  GO criteria: Alt-season beats alt-winter by > 50% Sharpe = {h5_go}')
print(f'  NO-GO criteria: No meaningful difference = {h5_nogo}')

In [ ]:
# ── Cell 15: Equity curves — gross and net, regime-shaded ────────
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Equity curve
ax = axes[0]
cum_gross = (1 + bt_filtered['gross']).cumprod()
cum_net = (1 + bt_filtered['net']).cumprod()
ax.plot(cum_gross.index, cum_gross.values, label='Gross', alpha=0.8, linewidth=1.5)
ax.plot(cum_net.index, cum_net.values, label='Net (4bps)', alpha=0.8, linewidth=1.5, color='orange')

# Shade alt-season/winter
if 'alt_season' in bt_with_regime.columns:
    season_mask = bt_with_regime['alt_season'].fillna(False).astype(bool)
    ymin, ymax = ax.get_ylim()
    for i in range(len(bt_with_regime)):
        if i > 0:
            color = 'lime' if season_mask.iloc[i] else 'red'
            ax.axvspan(bt_with_regime.index[i-1], bt_with_regime.index[i], alpha=0.05, color=color)

ax.set_title('L/S Portfolio Equity Curve (Regime-Shaded: Green=Alt-Season, Red=Alt-Winter)')
ax.set_ylabel('Cumulative Return')
ax.legend()
ax.grid(alpha=0.2)

# Drawdown
ax2 = axes[1]
peak_net = cum_net.cummax()
dd_net = (cum_net - peak_net) / peak_net
ax2.fill_between(dd_net.index, dd_net.values, 0, alpha=0.4, color='red')
ax2.set_title('Net Drawdown')
ax2.set_ylabel('Drawdown')
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 16: Risk analysis ───────────────────────────────────────
print('='*60)
print('Risk Analysis')
print('='*60)
print()

# Max drawdown
print(f'Max drawdown (gross): {mdd_gross:.2%}')
print(f'Max drawdown (net):   {mdd_net:.2%}')

# Turnover analysis
print(f'\nTurnover Analysis:')
print(f'  Mean turnover/rebalance: {bt_filtered["turnover"].mean():.1%}')
print(f'  Median turnover:         {bt_filtered["turnover"].median():.1%}')
print(f'  Max turnover:            {bt_filtered["turnover"].max():.1%}')
print(f'  Mean positions changed:  {bt_filtered["n_changed"].mean():.1f}')

if bt_filtered['turnover'].mean() > 0.8:
    print('  ⚠ WARNING: Turnover > 80% — strategy may be too expensive!')

# Per-asset PnL attribution
print('\nPer-Asset PnL Attribution:')
asset_pnl = {}
for sym in universe_data.keys():
    sym_long = pf_filtered[(pf_filtered['symbol'] == sym) & (pf_filtered['side'] == 'long')]
    sym_short = pf_filtered[(pf_filtered['symbol'] == sym) & (pf_filtered['side'] == 'short')]
    
    total_ret = 0.0
    n_trades = 0
    for _, row in sym_long.iterrows():
        ts = row['ts']
        if sym in universe_data:
            idx = universe_data[sym].index.get_indexer([ts], method='nearest')[0]
            if idx + 1 < len(universe_data[sym]):
                ret = (universe_data[sym]['close'].iloc[idx+1] / universe_data[sym]['close'].iloc[idx]) - 1
                total_ret += ret
                n_trades += 1
    for _, row in sym_short.iterrows():
        ts = row['ts']
        if sym in universe_data:
            idx = universe_data[sym].index.get_indexer([ts], method='nearest')[0]
            if idx + 1 < len(universe_data[sym]):
                ret = (universe_data[sym]['close'].iloc[idx+1] / universe_data[sym]['close'].iloc[idx]) - 1
                total_ret -= ret  # Short
                n_trades += 1
    asset_pnl[sym] = {'total_ret': total_ret, 'n_trades': n_trades,
                      'avg_ret': total_ret / n_trades if n_trades > 0 else 0}

pnl_df = pd.DataFrame(asset_pnl).T.sort_values('total_ret', ascending=False)
print(f'{"Symbol":<12} {"Total Ret":>10} {"N Trades":>10} {"Avg Ret":>10}')
print('-' * 45)
for sym, row in pnl_df.iterrows():
    if row['n_trades'] > 0:
        print(f'{sym:<12} {row["total_ret"]:>10.4%} {int(row["n_trades"]):>10d} {row["avg_ret"]:>10.4%}')

# Bar chart of per-asset PnL
fig, ax = plt.subplots(figsize=(14, 4))
active = pnl_df[pnl_df['n_trades'] > 0]
colors = ['lime' if x > 0 else 'red' for x in active['total_ret']]
ax.bar(active.index, active['total_ret'], color=colors, alpha=0.7)
ax.set_title('Per-Asset Cumulative PnL Attribution')
ax.set_ylabel('Total Return')
ax.axhline(0, color='white', linewidth=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 17: Final Verdict ───────────────────────────────────────
print('='*70)
print('  CROSS-SECTIONAL FACTOR NEUTRAL WITH FUNDING SQUEEZE — FINAL VERDICT')
print('='*70)
print()

verdicts = {
    'H1': {'desc': 'Momentum factor IC > 0', 'verdict': h1_verdict,
           'detail': f'Mean IC={mean_ic:.4f}, t={t_stat:.2f}'},
    'H2': {'desc': 'Funding extremes predict reversals', 'verdict': h2_verdict,
           'detail': f'Contrarian WR={combined_wr:.1%}'},
    'H3': {'desc': 'Funding filter improves L/S', 'verdict': h3_verdict,
           'detail': f'Sharpe improvement={improvement:+.1f}%'},
    'H4': {'desc': 'Net Sharpe > 0.5', 'verdict': h4_verdict,
           'detail': f'Net Sharpe={sr_filt_net:.3f}'},
    'H5': {'desc': 'Alt-season regime gate', 'verdict': h5_verdict,
           'detail': f'Season Sharpe={sr_season:.3f}, Winter={sr_winter:.3f}' if not np.isnan(sr_season) else 'Insufficient data'},
}

print(f'{"H":<4} {"Hypothesis":<40} {"Verdict":<12} {"Detail"}')
print('-' * 90)
n_go = 0
n_total = len(verdicts)
for h, v in verdicts.items():
    marker = '✅' if v['verdict'] == 'GO' else ('❌' if v['verdict'] == 'NO-GO' else '⚠️')
    print(f'{h:<4} {v["desc"]:<40} {marker} {v["verdict"]:<8} {v["detail"]}')
    if v['verdict'] == 'GO':
        n_go += 1

print()
print(f'Score: {n_go}/{n_total} hypotheses GO')
print()

# Overall verdict
if n_go >= 4:
    overall = 'GO — Proceed to parameter optimization and live testing'
elif n_go >= 3:
    overall = 'CONDITIONAL GO — Address failing hypotheses before production'
elif n_go >= 2:
    overall = 'BORDERLINE — Significant structural issues, needs redesign'
else:
    overall = 'NO-GO — Cross-sectional factor neutral does not work on this data'

print(f'OVERALL: {overall}')
print()
print('Key Statistics:')
print(f'  Gross Sharpe:  {sr_filt_gross:.3f}')
print(f'  Net Sharpe:    {sr_filt_net:.3f}')
print(f'  Max Drawdown:  {mdd_net:.2%}')
print(f'  Turnover:      {bt_filtered["turnover"].mean():.1%}')
print(f'  Universe size: {len(universe_data)} symbols')
print(f'  Data period:   {bt_filtered.index[0].date()} to {bt_filtered.index[-1].date()}')

In [ ]:
# ── Extract compact verdict summary ──
print("CROSS-SECTIONAL VERDICTS:")
for h, v in verdicts.items():
    marker = '✅' if v['verdict'] == 'GO' else ('❌' if v['verdict'] == 'NO-GO' else '⚠️')
    print(f"  {h}: {marker} {v['verdict']} | {v['detail']}")
print(f"\nKEY STATS:")
print(f"  Gross Sharpe: {sr_filt_gross:.3f}")
print(f"  Net Sharpe:   {sr_filt_net:.3f}")
print(f"  Max DD:       {mdd_net:.2%}")
print(f"  Turnover:     {bt_filtered['turnover'].mean():.1%}")
print(f"  Universe:     {len(universe_data)} symbols")
print(f"  Period:       {bt_filtered.index[0].date()} to {bt_filtered.index[-1].date()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FIX: Mean-Reversion Factors + Daily Rebalance + Top-10 Liquid
# ══════════════════════════════════════════════════════════════════
print("=" * 70)
print("CROSS-SECTIONAL FIX: MR + Daily Rebalance + Reduced Universe")
print("=" * 70)

# Step 1: Identify top-10 most liquid symbols by median dollar volume
median_dvols = {}
for sym, df in universe_data.items():
    median_dvols[sym] = (df['volume'] * df['close']).median()
top10 = sorted(median_dvols, key=median_dvols.get, reverse=True)[:10]
print(f"\nTop 10 liquid: {top10}")
print(f"Dollar vol range: ${median_dvols[top10[0]]:,.0f} to ${median_dvols[top10[-1]]:,.0f}")

# Step 2: Build DAILY panel (resample 4h → 1D to reduce turnover)
daily_panels = []
for sym in top10:
    df = universe_data[sym].copy()
    daily = df.resample('1D').agg({
        'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last',
        'volume': 'sum', 'taker_buy_base': 'sum'
    }).dropna()
    
    factors = pd.DataFrame(index=daily.index)
    factors['symbol'] = sym
    factors['close'] = daily['close']
    
    # MR momentum (NEGATE! — original IC was -0.023 → contrarian works better)
    factors['mr_mom_5d'] = -daily['close'].pct_change(5)   # 5-day MR
    factors['mr_mom_3d'] = -daily['close'].pct_change(3)   # 3-day MR
    
    # TFI (still directional — higher TFI = buying pressure)
    factors['tfi'] = (daily['taker_buy_base'] / daily['volume'].replace(0, np.nan)).ewm(span=3).mean()
    
    # Forward 1-day return
    factors['fwd_1d'] = daily['close'].pct_change(1).shift(-1)
    
    # Funding z-score
    if sym in universe_funding:
        fr = universe_funding[sym][['fundingRate']].copy()
        fr_daily = fr.resample('1D').last().ffill()
        factors = factors.join(fr_daily, how='left')
        factors['fundingRate'] = factors['fundingRate'].ffill()
        fr_mean = factors['fundingRate'].rolling(30, min_periods=10).mean()
        fr_std = factors['fundingRate'].rolling(30, min_periods=10).std()
        factors['funding_z'] = (factors['fundingRate'] - fr_mean) / fr_std.replace(0, np.nan)
    else:
        factors['funding_z'] = np.nan
    
    daily_panels.append(factors)

panel_daily = pd.concat(daily_panels)
print(f"\nDaily panel: {len(panel_daily)} rows, {panel_daily['symbol'].nunique()} symbols")

# Step 3: MR Factor IC test (the critical check)
print(f"\n{'='*70}")
print("FACTOR IC CHECK (Daily)")
print(f"{'='*70}")

mr_ics_5d = []
mr_ics_3d = []
tfi_ics = []
for ts in panel_daily.index.unique():
    group = panel_daily.loc[ts]
    if isinstance(group, pd.Series):
        continue
    g = group.dropna(subset=['mr_mom_5d', 'fwd_1d'])
    if len(g) >= 6:
        rho, _ = stats.spearmanr(g['mr_mom_5d'], g['fwd_1d'])
        mr_ics_5d.append(rho)
    g3 = group.dropna(subset=['mr_mom_3d', 'fwd_1d'])
    if len(g3) >= 6:
        rho3, _ = stats.spearmanr(g3['mr_mom_3d'], g3['fwd_1d'])
        mr_ics_3d.append(rho3)
    gt = group.dropna(subset=['tfi', 'fwd_1d'])
    if len(gt) >= 6:
        rhot, _ = stats.spearmanr(gt['tfi'], gt['fwd_1d'])
        tfi_ics.append(rhot)

for name, ics_arr in [('MR Momentum 5d', mr_ics_5d), ('MR Momentum 3d', mr_ics_3d), ('TFI', tfi_ics)]:
    ics_arr = np.array(ics_arr)
    if len(ics_arr) > 0:
        ic_mean = ics_arr.mean()
        ic_std = ics_arr.std()
        t = ic_mean / (ic_std / np.sqrt(len(ics_arr))) if ic_std > 0 else 0
        pct_pos = (ics_arr > 0).mean()
        print(f"  {name:20s}: IC={ic_mean:+.4f}, t={t:.2f}, %pos={pct_pos:.1%}, n={len(ics_arr)}")

print(f"  (Original 4h momentum IC was {mean_ic:+.4f})")

# Step 4: Portfolio construction — sweep configurations
print(f"\n{'='*70}")
print("PORTFOLIO SWEEP")
print(f"{'='*70}")

configs = [
    ('MR5d pure',         'mr_mom_5d', False, 3, 3),
    ('MR5d+TFI',          'mr_mom_5d', False, 3, 3),  # composite
    ('MR3d pure',         'mr_mom_3d', False, 3, 3),
    ('MR5d funding',      'mr_mom_5d', True,  3, 3),
    ('MR5d+TFI funding',  'mr_mom_5d', True,  3, 3),  # composite + funding
    ('MR5d wide',         'mr_mom_5d', False, 2, 2),
    ('MR5d+TFI wide',     'mr_mom_5d', False, 2, 2),  # composite, 2 each side
]

all_results = []

for cfg_name, mom_col, use_funding, n_long, n_short in configs:
    use_composite = '+TFI' in cfg_name
    portfolios = []
    
    for ts in panel_daily.index.unique():
        group = panel_daily.loc[ts]
        if isinstance(group, pd.Series):
            continue
        g = group.dropna(subset=[mom_col, 'tfi'])
        if len(g) < 6:
            continue
        
        g = g.copy()
        g['mr_rank'] = g[mom_col].rank(pct=True)
        if use_composite:
            g['tfi_rank'] = g['tfi'].rank(pct=True)
            g['score'] = g[['mr_rank', 'tfi_rank']].mean(axis=1)
        else:
            g['score'] = g['mr_rank']
        g = g.sort_values('score', ascending=False)
        
        if use_funding:
            long_cands = g.head(n_long * 3)
            longs = long_cands[long_cands['funding_z'] < -0.5].head(n_long)
            short_cands = g.tail(n_short * 3)
            shorts = short_cands[short_cands['funding_z'] > 0.5].head(n_short)
        else:
            longs = g.head(n_long)
            shorts = g.tail(n_short)
        
        for _, r in longs.iterrows():
            portfolios.append({'ts': ts, 'symbol': r['symbol'], 'side': 'long', 'fwd': r['fwd_1d']})
        for _, r in shorts.iterrows():
            portfolios.append({'ts': ts, 'symbol': r['symbol'], 'side': 'short', 'fwd': r['fwd_1d']})
    
    pf_df = pd.DataFrame(portfolios)
    if pf_df.empty:
        print(f"  {cfg_name}: No trades"); continue
    
    # Backtest
    daily_rets = []
    prev_pos = set()
    for ts in sorted(pf_df['ts'].unique()):
        sub = pf_df[pf_df['ts'] == ts]
        longs_fwd = sub[sub['side'] == 'long']['fwd'].dropna()
        shorts_fwd = sub[sub['side'] == 'short']['fwd'].dropna()
        
        l_ret = longs_fwd.mean() if len(longs_fwd) > 0 else 0
        s_ret = -shorts_fwd.mean() if len(shorts_fwd) > 0 else 0
        gross = (l_ret + s_ret) / 2
        
        # Turnover
        curr_pos = set(zip(sub['symbol'], sub['side']))
        if prev_pos:
            changed = len(curr_pos - prev_pos) + len(prev_pos - curr_pos)
            total = max(len(curr_pos) + len(prev_pos), 1)
            turnover = changed / total
        else:
            turnover = 1.0
        prev_pos = curr_pos
        
        cost = turnover * 4 * 2 / 10000  # 4bps each way
        net = gross - cost
        daily_rets.append({'ts': ts, 'gross': gross, 'net': net, 'turnover': turnover, 'n': len(sub)})
    
    bt_df = pd.DataFrame(daily_rets).set_index('ts')
    sr_g = bt_df['gross'].mean() / bt_df['gross'].std() * np.sqrt(365) if bt_df['gross'].std() > 0 else 0
    sr_n = bt_df['net'].mean() / bt_df['net'].std() * np.sqrt(365) if bt_df['net'].std() > 0 else 0
    tot_g = bt_df['gross'].sum() * 100
    tot_n = bt_df['net'].sum() * 100
    cum_net = bt_df['net'].cumsum()
    mdd_fix = (cum_net - cum_net.cummax()).min() * 100
    wr_fix = (bt_df['net'] > 0).mean() * 100
    avg_turnover = bt_df['turnover'].mean()
    
    print(f"  {cfg_name:25s}: SR_g={sr_g:+.3f} SR_n={sr_n:+.3f} Tot_n={tot_n:+.1f}% "
          f"WR={wr_fix:.0f}% MDD={mdd_fix:.1f}% TO={avg_turnover:.1%} days={len(bt_df)}")
    
    all_results.append({
        'config': cfg_name, 'sr_gross': sr_g, 'sr_net': sr_n,
        'total_net': tot_n, 'wr': wr_fix, 'mdd': mdd_fix,
        'turnover': avg_turnover, 'days': len(bt_df)
    })

# Compare with original
print(f"\n{'='*70}")
print("COMPARISON vs ORIGINAL")
print(f"{'='*70}")
print(f"  Original (4h momentum):    SR_g={sr_filt_gross:+.3f} SR_n={sr_filt_net:+.3f} "
      f"TO={bt_filtered['turnover'].mean():.1%}")

res_df = pd.DataFrame(all_results).sort_values('sr_net', ascending=False)
print(f"\n  Best daily MR config: {res_df.iloc[0]['config']}")
print(f"    SR_g={res_df.iloc[0]['sr_gross']:+.3f}, SR_n={res_df.iloc[0]['sr_net']:+.3f}, "
      f"Total={res_df.iloc[0]['total_net']:+.1f}%, MDD={res_df.iloc[0]['mdd']:.1f}%")

best_net = res_df.iloc[0]['sr_net']
if best_net > 0.5:
    verdict = f"GO — Net Sharpe {best_net:.3f} > 0.5"
elif best_net > 0:
    verdict = f"MARGINAL — Net Sharpe {best_net:.3f} positive but < 0.5"
else:
    verdict = f"STILL NO-GO — Net Sharpe {best_net:.3f}"
print(f"\n  CS FIX VERDICT: {verdict}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# WALK-FORWARD VALIDATION: Cross-Sectional MR+TFI
# ══════════════════════════════════════════════════════════════════
print("=" * 70)
print("WALK-FORWARD VALIDATION: Cross-Sectional MR+TFI L/S (Daily)")
print("=" * 70)

# Rebuild daily panel (same as fix cell)
median_dvols = {}
for sym, df in universe_data.items():
    median_dvols[sym] = (df['volume'] * df['close']).median()
top10 = sorted(median_dvols, key=median_dvols.get, reverse=True)[:10]

daily_panels_wf = []
for sym in top10:
    df = universe_data[sym].copy()
    daily = df.resample('1D').agg({
        'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last',
        'volume': 'sum', 'taker_buy_base': 'sum'
    }).dropna()
    factors = pd.DataFrame(index=daily.index)
    factors['symbol'] = sym
    factors['close'] = daily['close']
    factors['mr_mom_5d'] = -daily['close'].pct_change(5)
    factors['tfi'] = (daily['taker_buy_base'] / daily['volume'].replace(0, np.nan)).ewm(span=3).mean()
    factors['fwd_1d'] = daily['close'].pct_change(1).shift(-1)
    daily_panels_wf.append(factors)

panel_wf = pd.concat(daily_panels_wf)

# Split at 60%
all_dates = sorted(panel_wf.index.unique())
split_date = all_dates[int(len(all_dates) * 0.6)]
print(f"Total days: {len(all_dates)}")
print(f"Train: {all_dates[0].date()} to {split_date.date()} ({int(len(all_dates)*0.6)} days)")
print(f"Test:  {split_date.date()} to {all_dates[-1].date()} ({len(all_dates) - int(len(all_dates)*0.6)} days)")

def run_cs_backtest(panel_subset, label):
    portfolios = []
    for ts in panel_subset.index.unique():
        group = panel_subset.loc[ts]
        if isinstance(group, pd.Series):
            continue
        g = group.dropna(subset=['mr_mom_5d', 'tfi'])
        if len(g) < 6:
            continue
        g = g.copy()
        g['mr_rank'] = g['mr_mom_5d'].rank(pct=True)
        g['tfi_rank'] = g['tfi'].rank(pct=True)
        g['score'] = g[['mr_rank', 'tfi_rank']].mean(axis=1)
        g = g.sort_values('score', ascending=False)
        longs = g.head(2)
        shorts = g.tail(2)
        for _, r in longs.iterrows():
            portfolios.append({'ts': ts, 'symbol': r['symbol'], 'side': 'long', 'fwd': r['fwd_1d']})
        for _, r in shorts.iterrows():
            portfolios.append({'ts': ts, 'symbol': r['symbol'], 'side': 'short', 'fwd': r['fwd_1d']})
    
    pf_df = pd.DataFrame(portfolios)
    if pf_df.empty:
        print(f"  {label}: No trades")
        return {}
    
    daily_rets = []
    prev_pos = set()
    for ts in sorted(pf_df['ts'].unique()):
        sub = pf_df[pf_df['ts'] == ts]
        l_fwd = sub[sub['side']=='long']['fwd'].dropna()
        s_fwd = sub[sub['side']=='short']['fwd'].dropna()
        l_ret = l_fwd.mean() if len(l_fwd) > 0 else 0
        s_ret = -s_fwd.mean() if len(s_fwd) > 0 else 0
        gross = (l_ret + s_ret) / 2
        curr = set(zip(sub['symbol'], sub['side']))
        if prev_pos:
            changed = len(curr - prev_pos) + len(prev_pos - curr)
            total_p = max(len(curr) + len(prev_pos), 1)
            turnover = changed / total_p
        else:
            turnover = 1.0
        prev_pos = curr
        cost = turnover * 4 * 2 / 10000
        net = gross - cost
        daily_rets.append({'ts': ts, 'gross': gross, 'net': net, 'turnover': turnover})
    
    bt = pd.DataFrame(daily_rets).set_index('ts')
    sr_g = bt['gross'].mean() / bt['gross'].std() * np.sqrt(365) if bt['gross'].std() > 0 else 0
    sr_n = bt['net'].mean() / bt['net'].std() * np.sqrt(365) if bt['net'].std() > 0 else 0
    tot_n = bt['net'].sum() * 100
    cum = bt['net'].cumsum()
    mdd = (cum - cum.cummax()).min() * 100
    wr = (bt['net'] > 0).mean() * 100
    to = bt['turnover'].mean()
    print(f"  {label}: {len(bt)} days | SR_g={sr_g:+.3f} | SR_n={sr_n:+.3f} | Tot={tot_n:+.1f}% | WR={wr:.0f}% | MDD={mdd:.1f}% | TO={to:.1%}")
    return {'sr_gross': sr_g, 'sr_net': sr_n, 'total': tot_n, 'wr': wr, 'mdd': mdd, 'days': len(bt), 'daily_rets': bt['net'].values}

train_panel = panel_wf[panel_wf.index <= split_date]
test_panel = panel_wf[panel_wf.index > split_date]

print(f"\n▸ Walk-Forward Results (MR5d+TFI wide, 2L/2S):")
train_res = run_cs_backtest(train_panel, 'TRAIN')
test_res = run_cs_backtest(test_panel, 'TEST ')
full_res = run_cs_backtest(panel_wf, 'FULL ')

if train_res.get('sr_net', 0) != 0:
    wf = test_res.get('sr_net', 0) / train_res['sr_net']
    print(f"\n  WF Ratio: {wf:.2f}")

# Bootstrap on full period daily returns
print(f"\n{'='*70}")
print("BOOTSTRAP (10,000 resamples of daily L/S returns)")
print(f"{'='*70}")
if 'daily_rets' in full_res and len(full_res['daily_rets']) >= 30:
    rets = full_res['daily_rets']
    n_boot = 10000
    np.random.seed(42)
    boot_sharpes = []
    boot_totals = []
    for _ in range(n_boot):
        sample = np.random.choice(rets, size=len(rets), replace=True)
        boot_sharpes.append((sample.mean() / sample.std()) * np.sqrt(365) if sample.std() > 0 else 0)
        boot_totals.append(sample.sum() * 100)
    boot_sharpes = np.array(boot_sharpes)
    boot_totals = np.array(boot_totals)
    print(f"  Sharpe 95% CI: [{np.percentile(boot_sharpes, 2.5):.2f}, {np.percentile(boot_sharpes, 97.5):.2f}]")
    print(f"  Total PnL 95% CI: [{np.percentile(boot_totals, 2.5):+.1f}%, {np.percentile(boot_totals, 97.5):+.1f}%]")
    print(f"  P(Sharpe > 0): {(boot_sharpes > 0).mean()*100:.1f}%")
    print(f"  P(Sharpe > 0.5): {(boot_sharpes > 0.5).mean()*100:.1f}%")
    print(f"  P(Total > 0): {(boot_totals > 0).mean()*100:.1f}%")